<a href="https://colab.research.google.com/github/sidhu2690/sid-archive/blob/main/deep-gradient-dynamics/verify_symmetry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import torch
import numpy as np

torch.set_default_dtype(torch.float64)

def make_data(seed=0, amp=0.3, noise=0.0, n_tr=2000):
  rng = np.random.default_rng(seed)
  x = rng.uniform(-np.pi, np.pi, n_tr)
  y = amp * np.sin(x) + noise * rng.standard_normal(n_tr)
  return torch.tensor(x), torch.tensor(y)


def train(amp, D, opt, steps=8000, noise=0.0, asym_init=False):
  x, y = make_data(0, amp, noise)
  w = torch.nn.Parameter(torch.tensor(1.5))

  if asym_init:
      init = np.random.default_rng(123).uniform(0.8, 1.2, D)
  else:
      init = np.full(D, 0.9)

  a = torch.nn.Parameter(torch.tensor(init))

  if opt == "sgd":
      optimizer = torch.optim.SGD([w, a], lr=0.25)
  else:
      optimizer = torch.optim.Adam([w, a], lr=3e-3)

  max_grad_dev = 0.0
  max_spread = 0.0

  log_at = {0, 1, 2, 10, 100, 1000, 4000, 7999}

  for t in range(steps + 1):
    optimizer.zero_grad()

    #model
    A = torch.prod(a)
    y_pred = A * torch.sin(w * x)

    #loss
    loss = torch.mean((y_pred - y)**2)
    loss.backward()

    #symmetry
    with torch.no_grad():
      grad_dev = (a.grad.max() - a.grad.min()).abs()
      spread = (a.max() - a.min()).abs()

      max_grad_dev = max(max_grad_dev, grad_dev.item())
      max_spread = max(max_spread, spread.item())

      if t in log_at:
        print(f"step={t:5d} | grad deviation={grad_dev.item():.3e} | scale spread={spread.item():.3e}")

      if t == steps:
          break

      optimizer.step()

  return max_grad_dev, max_spread, loss.item()

In [9]:
print("EQUAL INITIALIZATION")
print("-" * 65)

for D in [1, 8, 64]:
  for opt in ["sgd", "adam"]:
    for amp in [0.3, 1.0]:
      gd, spread, loss = train(amp, D, opt)
      print(f"D={D:2d} | {opt:4s} | amp={amp} | max grad dev={gd:.3e} | max spread={spread:.3e} | loss={loss:.3e}")

EQUAL INITIALIZATION
-----------------------------------------------------------------
step=    0 | grad deviation=0.000e+00 | scale spread=0.000e+00
step=    1 | grad deviation=0.000e+00 | scale spread=0.000e+00
step=    2 | grad deviation=0.000e+00 | scale spread=0.000e+00
step=   10 | grad deviation=0.000e+00 | scale spread=0.000e+00
step=  100 | grad deviation=0.000e+00 | scale spread=0.000e+00
step= 1000 | grad deviation=0.000e+00 | scale spread=0.000e+00
step= 4000 | grad deviation=0.000e+00 | scale spread=0.000e+00
step= 7999 | grad deviation=0.000e+00 | scale spread=0.000e+00
D= 1 | sgd  | amp=0.3 | max grad dev=0.000e+00 | max spread=0.000e+00 | loss=2.909e-31
step=    0 | grad deviation=0.000e+00 | scale spread=0.000e+00
step=    1 | grad deviation=0.000e+00 | scale spread=0.000e+00
step=    2 | grad deviation=0.000e+00 | scale spread=0.000e+00
step=   10 | grad deviation=0.000e+00 | scale spread=0.000e+00
step=  100 | grad deviation=0.000e+00 | scale spread=0.000e+00
step= 1

In [15]:
print("UNEQUAL INITIALISATION CONTROL")
print("-" * 65)

for D in [4, 8, 64]:
  x, y = make_data(0, 0.3)

  w = torch.tensor(1.5, requires_grad=True)
  init = np.random.default_rng(123).uniform(0.8, 1.2, D)
  a = torch.tensor(init, requires_grad=True)

  loss = torch.mean((torch.prod(a) * torch.sin(w*x) - y)**2)
  loss.backward()
  print(f"D={D:2d} | gradient min={a.grad.min().item():.6f} | gradient max={a.grad.max().item():.6f}")

UNEQUAL INITIALISATION CONTROL
-----------------------------------------------------------------
D= 4 | gradient min=0.341407 | gradient max=0.445887
D= 8 | gradient min=0.344370 | gradient max=0.490165
D=64 | gradient min=0.130546 | gradient max=0.190778


In [11]:
print("NOISY DATA CONTROL")
print("-" * 65)

train(0.3, 64, "adam", steps=1000, noise=0.02)

NOISY DATA CONTROL
-----------------------------------------------------------------
step=    0 | grad deviation=0.000e+00 | scale spread=0.000e+00
step=    1 | grad deviation=0.000e+00 | scale spread=0.000e+00
step=    2 | grad deviation=0.000e+00 | scale spread=0.000e+00
step=   10 | grad deviation=0.000e+00 | scale spread=0.000e+00
step=  100 | grad deviation=0.000e+00 | scale spread=0.000e+00
step= 1000 | grad deviation=0.000e+00 | scale spread=0.000e+00


(0.0, 0.0, 0.00039727143762072735)